# A)Data Retrieval and Preprocessing

## Data Retrieval

In [28]:
!pip install kagglehub

In [29]:
from pathlib import Path
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    # 1) Look for local file first: ../raw_data/dataset.csv
    base_dir = Path.cwd().parent          # one level up from notebook
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    # 2) If not found, download from Kaggle once via kagglehub
    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",  # main CSV name on Kaggle
    )

    # Save for future offline use
    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

df = load_cashflow_data()

Loading local file: /Users/anton/code/DERNTOAN/cf_copilot/raw_data/dataset.csv


## A.1)Feature Extraction and Engineering

### A.1.1) Quick Look at Data Structure

In [30]:
df.head(2)

,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,invoice_currency,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen
0,U001,0200769623,WAL-MAR corp,2020-02-11 00:00:00,2020.0,1.930438e+09,2020-01-26,20200125,20200126,20200210.0,USD,RV,1.0,NaN,54273.28,20200126.0,NAH4,1.930438e+09,0
1,U001,0200980828,BEN E,2019-08-08 00:00:00,2019.0,1.929646e+09,2019-07-22,20190722,20190722,20190811.0,USD,RV,1.0,NaN,79656.60,20190722.0,NAD1,1.929646e+09,0


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   business_code           50000 non-null  object 
 1   cust_number             50000 non-null  object 
 2   name_customer           50000 non-null  object 
 3   clear_date              40000 non-null  object 
 4   buisness_year           50000 non-null  float64
 5   doc_id                  50000 non-null  float64
 6   posting_date            50000 non-null  object 
 7   document_create_date    50000 non-null  int64  
 8   document_create_date.1  50000 non-null  int64  
 9   due_in_date             50000 non-null  float64
 10  invoice_currency        50000 non-null  object 
 11  document type           50000 non-null  object 
 12  posting_id              50000 non-null  float64
 13  area_business           0 non-null      float64
 14  total_open_amount       50000 non-null

Checking values in all the columns

### A.1.2)Peeking into each of the columns

In [32]:
df['business_code'].value_counts()

business_code
U001    45359
CA02     3917
U013      573
U002      135
U005       11
U007        5
Name: count, dtype: int64

In [33]:
print(df['clear_date'].value_counts())
print(df['clear_date'].isnull().value_counts())

clear_date
2019-11-12 00:00:00    309
2019-09-03 00:00:00    249
2019-10-15 00:00:00    246
2019-11-01 00:00:00    242
2019-02-19 00:00:00    241
                      ... 
2020-02-16 00:00:00      1
2019-08-31 00:00:00      1
2020-01-06 00:00:00      1
2020-02-29 00:00:00      1
2019-07-20 00:00:00      1
Name: count, Length: 403, dtype: int64
clear_date
False    40000
True     10000
Name: count, dtype: int64


name_customer -> some names are repeated

clear_date -> False 40000, True 1000

business_data -> only 2019,2020 data

doc_id -> some repition as no missing id

invoice_currency -> USD,CAD

posting_date -> 506 posting dates

## A.2)Data Preprocessing and Wrangling

### A.2.1)Data Cleaning

Checking for duplicates

In [34]:
df.duplicated().value_counts()

False    48839
True      1161
Name: count, dtype: int64

In [35]:
df['document_create_date'] = pd.to_datetime(df.document_create_date, format="%Y%m%d")
df['document_create_date'].median

<bound method Series.median of 0       2020-01-25
1       2019-07-22
2       2019-09-14
3       2020-03-30
4       2019-11-13
           ...    
49995   2020-04-17
49996   2019-08-14
49997   2020-02-18
49998   2019-11-26
49999   2019-01-05
Name: document_create_date, Length: 50000, dtype: datetime64[ns]>

we can see 1161 datas duplicated.

## A.3)Feature Scaling & Selection

# Sliding window


In [36]:
CURRENT_DATE = pd.to_datetime(20200125, format="%Y%m%d")
CURRENT_DATE

df = df.dropna(subset=["clear_date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   business_code           40000 non-null  object        
 1   cust_number             40000 non-null  object        
 2   name_customer           40000 non-null  object        
 3   clear_date              40000 non-null  object        
 4   buisness_year           40000 non-null  float64       
 5   doc_id                  40000 non-null  float64       
 6   posting_date            40000 non-null  object        
 7   document_create_date    40000 non-null  datetime64[ns]
 8   document_create_date.1  40000 non-null  int64         
 9   due_in_date             40000 non-null  float64       
 10  invoice_currency        40000 non-null  object        
 11  document type           40000 non-null  object        
 12  posting_id              40000 non-null  float64    

### 1. Convert target to bucket 

In [ ]:
import numpy as np

df["days_to_payment"] = (pd.to_datetime(df["clear_date"]) - CURRENT_DATE).dt.days

# Convert to weekly buckets
df["week_bucket"] = np.ceil(df["days_to_payment"] / 7).astype(int)
df["week_bucket"] = df["week_bucket"].clip(lower=1)
df.loc[df["week_bucket"] > 6, "week_bucket"] = 7
df

,business_code,cust_number,name_customer,clear_date,buisness_year,doc_id,posting_date,document_create_date,document_create_date.1,due_in_date,...,document type,posting_id,area_business,total_open_amount,baseline_create_date,cust_payment_terms,invoice_id,isOpen,days_to_payment,week_bucket
0,U001,0200769623,WAL-MAR corp,2020-02-11 00:00:00,2020.0,1.930438e+09,2020-01-26,2020-01-25,20200126,20200210.0,...,RV,1.0,NaN,54273.28,20200126.0,NAH4,1.930438e+09,0,17,3
1,U001,0200980828,BEN E,2019-08-08 00:00:00,2019.0,1.929646e+09,2019-07-22,2019-07-22,20190722,20190811.0,...,RV,1.0,NaN,79656.60,20190722.0,NAD1,1.929646e+09,0,-170,1
2,U001,0200792734,MDV/ trust,2019-12-30 00:00:00,2019.0,1.929874e+09,2019-09-14,2019-09-14,20190914,20190929.0,...,RV,1.0,NaN,2253.86,20190914.0,NAA8,1.929874e+09,0,-26,1
4,U001,0200769623,WAL-MAR foundation,2019-11-25 00:00:00,2019.0,1.930148e+09,2019-11-13,2019-11-13,20191113,20191128.0,...,RV,1.0,NaN,33133.29,20191113.0,NAH4,1.930148e+09,0,-61,1
5,CA02,0140106181,THE corporation,2019-12-04 00:00:00,2019.0,2.960581e+09,2019-09-20,2019-09-20,20190920,20191004.0,...,RV,1.0,NaN,22225.84,20190924.0,CA10,2.960581e+09,0,-52,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49994,U001,0200762301,C&S WH trust,2019-07-25 00:00:00,2019.0,1.929601e+09,2019-07-10,2019-07-09,20190710,20190725.0,...,RV,1.0,NaN,84780.40,20190710.0,NAC6,1.929601e+09,0,-184,1
49996,U001,0200769623,WAL-MAR co,2019-09-03 00:00:00,2019.0,1.929744e+09,2019-08-15,2019-08-14,20190815,20190830.0,...,RV,1.0,NaN,6766.54,20190815.0,NAH4,1.929744e+09,0,-144,1
49997,U001,0200772595,SAFEW associates,2020-03-05 00:00:00,2020.0,1.930537e+09,2020-02-19,2020-02-18,20200219,20200305.0,...,RV,1.0,NaN,6120.86,20200219.0,NAA8,1.930537e+09,0,40,6
49998,U001,0200726979,BJ'S llc,2019-12-12 00:00:00,2019.0,1.930199e+09,2019-11-27,2019-11-26,20191127,20191212.0,...,RV,1.0,NaN,63.48,20191127.0,NAA8,1.930199e+09,0,-44,1


### 2. use a sliding window to create the data adn train the model

In [ ]:
from dateutil.relativedelta import relativedelta

window_months = 3
horizon_weeks = 6

df["posting_date"] = pd.to_datetime(df["posting_date"])
df = df.sort_values("posting_date").reset_index(drop=True)

min_date = df["posting_date"].min()
max_date = df["posting_date"].max()

current = min_date + relativedelta(months=window_months)

while current + pd.Timedelta(weeks=horizon_weeks) <= max_date:
    window_start = current - relativedelta(months=window_months)

    # Convert target to weekly buckets
    df["days_to_payment"] = (pd.to_datetime(df["clear_date"]) - current).dt.days
    df["week_bucket"] = np.ceil(df["days_to_payment"] / 7).astype(int)
    df["week_bucket"] = df["week_bucket"].clip(lower=1)
    df.loc[df["week_bucket"] > 6, "week_bucket"] = 7

    train = df[(df["posting_date"] >= window_start) & (df["posting_date"] < current)]
    test = df[(df["posting_date"] >= current) & (df["posting_date"] < current + pd.Timedelta(weeks=horizon_weeks))]

    print(f"Train: {window_start.date()} to {current.date()} ({len(train)} rows)")
    print(f"Test:  {current.date()} to {(current + pd.Timedelta(weeks=horizon_weeks)).date()} ({len(test)} rows)")
    print()

    # Train your model here
    # model.fit(train[features], train["week_bucket"])
    # probas = model.predict_proba(test[features])

    current += pd.Timedelta(weeks=horizon_weeks)  # stride forward

Train: 2018-12-30 to 2019-03-30 (8668 rows)
Test:  2019-03-30 to 2019-05-11 (4418 rows)

Train: 2019-02-11 to 2019-05-11 (9338 rows)
Test:  2019-05-11 to 2019-06-22 (4134 rows)

Train: 2019-03-22 to 2019-06-22 (9412 rows)
Test:  2019-06-22 to 2019-08-03 (4097 rows)

Train: 2019-05-03 to 2019-08-03 (9086 rows)
Test:  2019-08-03 to 2019-09-14 (4186 rows)

Train: 2019-06-14 to 2019-09-14 (9173 rows)
Test:  2019-09-14 to 2019-10-26 (4312 rows)

Train: 2019-07-26 to 2019-10-26 (9268 rows)
Test:  2019-10-26 to 2019-12-07 (3887 rows)

Train: 2019-09-07 to 2019-12-07 (8900 rows)
Test:  2019-12-07 to 2020-01-18 (2627 rows)



# B)ALGO Testing

# C)Model Eval and Tuning

# D)Pipeline